#Q1

##Imports

In [1]:
!pip install sentence-transformers rouge-score bert-score nltk

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.0 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=5790f023892477c55239a8e1b5fa46be26633134659c40f7c4d947796ad342aa
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


##Main code

In [9]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score
from tqdm import tqdm

# Paths
TRAIN_PATH = '/content/drive/MyDrive/P1_data/trac2_CONVT_train.csv'
DEV_PATH = '/content/drive/MyDrive/P1_data/trac2_CONVT_dev.csv'
TEST_PATH = '/content/drive/MyDrive/P1_data/trac2_CONVT_test.csv'

# Output paths
DEV_OUTPUT = '/content/drive/MyDrive/P2_Outputs/generations_corpus_dev.csv'
TEST_OUTPUT = '/content/drive/MyDrive/P2_Outputs/generations_corpus.csv'

class CorpusBasedChatbot:

    def __init__(self, w1=0.5, w2=0.2, w3=0.2, w4=0.1):
        self.w1 = w1
        self.w2 = w2
        self.w3 = w3
        self.w4 = w4
        self.num_polarity_classes = 4

        print("SentenceTransformer model loading")
        self.model = SentenceTransformer('all-MiniLM-L6-v2')

        self.train_data = None
        self.train_embeddings = None
        self.train_polarity_onehot = None

    def one_hot_encode_polarity(self, polarity_values):
        n_samples = len(polarity_values)
        one_hot = np.zeros((n_samples, self.num_polarity_classes))
        one_hot[np.arange(n_samples), polarity_values.astype(int)] = 1
        return one_hot

    def load_and_prepare_data(self, train_path):

        print("Load training data")
        self.train_data = pd.read_csv(train_path)
        self.train_data['emotion_norm'] = self.train_data['Emotion'] / 5.0
        self.train_data['empathy_norm'] = self.train_data['Empathy'] / 5.0

        self.train_polarity_onehot = self.one_hot_encode_polarity(
            self.train_data['EmotionalPolarity'].values
        )

        print("Generate embeddings for training data")
        self.train_embeddings = self.model.encode(
            self.train_data['text'].tolist(),
            show_progress_bar=True,
            convert_to_numpy=True,
            batch_size=32
        )

        print(f"Training data: {len(self.train_data)} utterances\n")

    def calculate_similarity(self, query_embedding, query_emotion, query_empathy,
                           query_polarity_onehot):

        s_text = cosine_similarity(
            query_embedding.reshape(1, -1),
            self.train_embeddings
        )[0]

        s_emotion = 1 - np.abs(self.train_data['emotion_norm'].values - query_emotion)
        s_empathy = 1 - np.abs(self.train_data['empathy_norm'].values - query_empathy)

        s_polarity = cosine_similarity(
            query_polarity_onehot.reshape(1, -1),
            self.train_polarity_onehot
        )[0]

        S_total = (self.w1 * s_text +
                   self.w2 * s_emotion +
                   self.w3 * s_empathy +
                   self.w4 * s_polarity)

        return S_total

    def retrieve_response(self, context_history, target_emotion, target_empathy,target_polarity, excluded_indices=None):

        context_text = " ".join(context_history)
        query_embedding = self.model.encode(context_text, convert_to_numpy=True)
        emotion_norm = target_emotion / 5.0
        empathy_norm = target_empathy / 5.0
        query_polarity_onehot = self.one_hot_encode_polarity(np.array([target_polarity]))[0]

        similarities = self.calculate_similarity(
            query_embedding, emotion_norm, empathy_norm, query_polarity_onehot
        )

        if excluded_indices is not None and len(excluded_indices) > 0:
            similarities_masked = similarities.copy()
            for idx in excluded_indices:
                similarities_masked[idx] = -np.inf
        else:
            similarities_masked = similarities

        most_similar_idx = np.argmax(similarities_masked)

        if np.isinf(similarities_masked[most_similar_idx]):
            most_similar_idx = np.argmax(similarities)

        retrieved_response = self.train_data.iloc[most_similar_idx]['text']

        return retrieved_response, most_similar_idx

    def generate_conversation(self, conversation_df, start_turn=6, num_turns=5,
                            has_labels=True):
        generations = []
        conversation_df = conversation_df.sort_values('turn_id')
        history = conversation_df[conversation_df['turn_id'] < start_turn]['text'].tolist()
        used_indices = set()

        for turn in range(start_turn, start_turn + num_turns):
            turn_data = conversation_df[conversation_df['turn_id'] == turn]
            has_emotion_cols = len(turn_data) > 0 and 'Emotion' in turn_data.columns

            if has_emotion_cols:
                target_emotion = turn_data.iloc[0]['Emotion']
                target_empathy = turn_data.iloc[0]['Empathy']
                target_polarity = turn_data.iloc[0]['EmotionalPolarity']
                gold_response = turn_data.iloc[0]['text'] if has_labels else None
            else:
                target_emotion = 2.5
                target_empathy = 2.5
                target_polarity = 1
                gold_response = None

            # Retrieve most similar response
            generated_response, selected_idx = self.retrieve_response(
                history, target_emotion, target_empathy, target_polarity,
                excluded_indices=used_indices
            )

            used_indices.add(selected_idx)

            generations.append({
                'id': conversation_df.iloc[0]['conversation_id'],
                'turn_number': turn,
                'generated_response': generated_response,
                'gold_text': gold_response,
                'target_emotion': target_emotion,
                'target_empathy': target_empathy,
                'target_polarity': target_polarity
            })

            history.append(generated_response)

        return generations


def evaluate_generations(generations_df):

    print("\nEvaluate on dev set")
    eval_df = generations_df[generations_df['gold_text'].notna()].copy()

    if len(eval_df) == 0:
        print("No gold references available for evaluation")
        return None

    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    bleu_scores = []
    smoothing = SmoothingFunction()

    for _, row in eval_df.iterrows():
        generated = row['generated_response']
        gold = row['gold_text']

        scores = rouge.score(gold, generated)
        rouge_scores['rouge1'].append(scores['rouge1'].fmeasure)
        rouge_scores['rouge2'].append(scores['rouge2'].fmeasure)
        rouge_scores['rougeL'].append(scores['rougeL'].fmeasure)

        reference = [gold.split()]
        candidate = generated.split()
        bleu = sentence_bleu(reference, candidate, smoothing_function=smoothing.method1)
        bleu_scores.append(bleu)

    P, R, F1 = bert_score(
        eval_df['generated_response'].tolist(),
        eval_df['gold_text'].tolist(),
        lang='en',
        verbose=False
    )

    metrics = {
        'ROUGE-1': np.mean(rouge_scores['rouge1']),
        'ROUGE-2': np.mean(rouge_scores['rouge2']),
        'ROUGE-L': np.mean(rouge_scores['rougeL']),
        'BLEU': np.mean(bleu_scores),
        'BERTScore-P': P.mean().item(),
        'BERTScore-R': R.mean().item(),
        'BERTScore-F1': F1.mean().item()
    }

    return metrics

def main():

    print("="*80)
    print("CORPUS-BASED CHATBOT")
    print("="*80)

    chatbot = CorpusBasedChatbot(w1=0.5, w2=0.2, w3=0.2, w4=0.1)
    chatbot.load_and_prepare_data(TRAIN_PATH)

    # Development Set
    print("="*80)
    print("DEVELOPMENT SET")
    print("="*80)

    dev_df = pd.read_csv(DEV_PATH, on_bad_lines='skip')
    unique_conv_ids = dev_df['conversation_id'].unique()
    all_dev_generations = []

    for conv_id in tqdm(unique_conv_ids, desc="Dev conversations"):
        conv_data = dev_df[dev_df['conversation_id'] == conv_id]
        generations = chatbot.generate_conversation(conv_data, start_turn=6, num_turns=5)
        all_dev_generations.extend(generations)

    dev_gen_df = pd.DataFrame(all_dev_generations)
    dev_gen_df[['id', 'turn_number', 'generated_response']].to_csv(DEV_OUTPUT, index=False)
    print(f"\nSaved: {DEV_OUTPUT}")

    # Evaluate
    metrics = evaluate_generations(dev_gen_df)
    if metrics:
        print("\n" + "="*80)
        print("DEVELOPMENT SET EVALUATION METRICS")
        print("="*80)
        for metric, value in metrics.items():
            print(f"{metric:15s}: {value:.4f}")

    # Test Set
    print("\n" + "="*80)
    print("TEST SET")
    print("="*80)

    test_df = pd.read_csv(TEST_PATH, on_bad_lines='skip')
    unique_test_conv_ids = test_df['conversation_id'].unique()
    all_test_generations = []

    for conv_id in tqdm(unique_test_conv_ids, desc="Test conversations"):
        conv_data = test_df[test_df['conversation_id'] == conv_id]
        generations = chatbot.generate_conversation(conv_data, start_turn=6, num_turns=5,
                                                   has_labels=False)
        all_test_generations.extend(generations)

    test_gen_df = pd.DataFrame(all_test_generations)
    test_gen_df[['id', 'turn_number', 'generated_response']].to_csv(TEST_OUTPUT, index=False)
    print(f"\nSaved: {TEST_OUTPUT}")

    # Sample Conversations for Q3
    print("\n" + "="*80)
    print("SAMPLE CONVERSATIONS FOR Q3 ANALYSIS")
    print("="*80)

    sample_conv_ids = unique_conv_ids[:5]

    for i, conv_id in enumerate(sample_conv_ids, 1):
        print(f"\n{'='*80}")
        print(f"Conversation {i} (ID: {conv_id})")
        print('='*80)
        conv_data = dev_df[dev_df['conversation_id'] == conv_id]

        print("\nContext (Turns 1-5):")
        for _, row in conv_data[conv_data['turn_id'] <= 5].iterrows():
            print(f"  Turn {row['turn_id']}: {row['text']}")

        generations = chatbot.generate_conversation(conv_data, start_turn=6, num_turns=10)

        print("\nGenerated (Turns 6-15):")
        for gen in generations:
            print(f"  Turn {gen['turn_number']}: {gen['generated_response']}")

    print("\n" + "="*80)
    print("GENERATION COMPLETE!")
    print("="*80)
    print(f"\nOutput files:")
    print(f"  - Development: {DEV_OUTPUT}")
    print(f"  - Test:        {TEST_OUTPUT}")
    print("="*80)

if __name__ == "__main__":
    main()

CORPUS-BASED CHATBOT
SentenceTransformer model loading
Load training data
Generate embeddings for training data


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

Training data: 11090 utterances

DEVELOPMENT SET


Dev conversations: 100%|██████████| 33/33 [00:05<00:00,  6.57it/s]



Saved: /content/drive/MyDrive/P2_Outputs/generations_corpus_dev.csv

Evaluate on dev set


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



DEVELOPMENT SET EVALUATION METRICS
ROUGE-1        : 0.1316
ROUGE-2        : 0.0195
ROUGE-L        : 0.1091
BLEU           : 0.0090
BERTScore-P    : 0.8550
BERTScore-R    : 0.8466
BERTScore-F1   : 0.8506

TEST SET


Test conversations: 100%|██████████| 67/67 [00:09<00:00,  6.88it/s]



Saved: /content/drive/MyDrive/P2_Outputs/generations_corpus.csv

SAMPLE CONVERSATIONS FOR Q3 ANALYSIS

Conversation 1 (ID: 68)

Context (Turns 1-5):
  Turn 1: Hello how are you?
  Turn 2: Hello! Fine. How are you?
  Turn 3: I'm good thanks for asking!
  Turn 4: That's good, so I guess we are supposed to discuss the article. I wonder, if my family or I were in a place where my very life was at risk, what would I do?
  Turn 5: That's a very scary thought to have honestly

Generated (Turns 6-15):
  Turn 6: I would do anything to protect my family, but I do not  go looking for trouble. I'm glad you can see it from an objective point of view because most can not.
  Turn 7: good talking to you. I hope it works out for the people affected. bye.
  Turn 8: Going back to what you said earlier. I feel bad for anybody family that has to go though this.
  Turn 9: Yeah! I hope the families that went through this are okay now though
  Turn 10: I just hope that the organizations are able to protect t